<a href="https://colab.research.google.com/github/eshikajindal24/UCS547_Accelerated_data_science/blob/main/UCS547_102303954_Assign_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ASSIGNMENT 4**

1. You are given a large NumPy array of size 5000000 ini8alized with random
values. Compute the following element-wise opera8on: f(x)=x2+3x+5, for
each element in the array and convert it into a CUDA kernel using Numba.
Compare performance difference of CPU with GPU.
a. Modify the kernel to use float32 and float64.

In [3]:
import numpy as np
import time
from numba import jit, cuda
def compute_cpu(arr):
    for i in range(len(arr)):
        arr[i] = arr[i]*arr[i] + 3*arr[i] + 5
    return arr

@jit(nopython=True)
def compute_numba(arr):
    for i in range(len(arr)):
        arr[i] = arr[i]*arr[i] + 3*arr[i] + 5
    return arr

@cuda.jit
def compute_gpu(arr):
    idx = cuda.grid(1)
    if idx < arr.size:
        x = arr[idx]
        arr[idx] = x*x + 3*x + 5

# Create array
arr = np.random.random(5000000)

# ---------------- CPU ----------------
arr_cpu = arr.copy()
start = time.time()
compute_cpu(arr_cpu)
print("CPU Time:", time.time() - start)

# ---------------- NUMBA CPU ----------------
arr_numba = arr.copy()
start = time.time()
compute_numba(arr_numba)   # first call (compile)
compute_numba(arr_numba)   # second call (fast)
print("Numba CPU Time:", time.time() - start)

# ---------------- GPU ----------------
arr_gpu = arr.copy()


d_arr = cuda.to_device(arr_gpu)

threads_per_block = 256
blocks_per_grid = (arr_gpu.size + threads_per_block - 1) // threads_per_block

start = time.time()
compute_gpu[blocks_per_grid, threads_per_block](d_arr)

d_arr.copy_to_host(arr_gpu)

print("GPU Time:", time.time() - start)


arr_float32 = np.random.random(5000000).astype(np.float32)

d_arr32 = cuda.to_device(arr_float32)
compute_gpu[blocks_per_grid, threads_per_block](d_arr32)
d_arr32.copy_to_host(arr_float32)


arr_float64 = np.random.random(5000000).astype(np.float64)

d_arr64 = cuda.to_device(arr_float64)
compute_gpu[blocks_per_grid, threads_per_block](d_arr64)
d_arr64.copy_to_host(arr_float64)

CPU Time: 4.076554536819458
Numba CPU Time: 0.12911486625671387
GPU Time: 0.10405874252319336


array([8.34110098, 5.804484  , 5.65761489, ..., 7.41517039, 5.28105121,
       7.38821914])

2. Implement and benchmark a 1-D histogram computa8on for 1 million
random values in Python using Numba. Compare different approaches (pure
Python, NumPy, and Numba-accelerated) and analyze performance and
correctness. (Ref: hSps://numba.pydata.org/numbaexamples/examples/density_es8ma8on/histogram/results.html )

In [6]:
#Pure Python
def histogram_python(arr, bins, min_val, max_val):
    hist = [0]*bins
    bin_width = (max_val - min_val) / bins

    for i in range(len(arr)):
        index = int((arr[i] - min_val) / bin_width)
        if index >= bins:
            index = bins - 1
        hist[index] += 1

    return hist

# NumPy
def histogram_numpy(arr, bins):
    hist, _ = np.histogram(arr, bins=bins)
    return hist

#Numba
@jit(nopython=True)
def histogram_numba(arr, bins, min_val, max_val):
    hist = np.zeros(bins)
    bin_width = (max_val - min_val) / bins

    for i in range(len(arr)):
        index = int((arr[i] - min_val) / bin_width)
        if index >= bins:
            index = bins - 1
        hist[index] += 1

    return hist

arr = np.random.random(1000000)

bins = 50
min_val = np.min(arr)
max_val = np.max(arr)

# -------- Python --------
start = time.time()
hist_py = histogram_python(arr, bins, min_val, max_val)
print("Python Time:", time.time() - start)

# -------- NumPy --------
start = time.time()
hist_np = histogram_numpy(arr, bins)
print("NumPy Time:", time.time() - start)

# -------- Numba --------
histogram_numba(arr, bins, min_val, max_val)
start = time.time()
hist_nb = histogram_numba(arr, bins, min_val, max_val)
print("Numba Time:", time.time() - start)

print("Python vs NumPy:", np.allclose(hist_py, hist_np))
print("NumPy vs Numba:", np.allclose(hist_np, hist_nb))

Python Time: 0.6354591846466064
NumPy Time: 0.02245330810546875
Numba Time: 0.0031447410583496094
Python vs NumPy: True
NumPy vs Numba: True


3. Write a func8on monte_carlo_pi(nsamples) that es8mates the value of π by
genera8ng random x, y coordinates between 0 and 1 and checking if they fall
inside a unit circle (x2 + y2 < 1).
a. Implement the func8on in pure Python first and later create a Numba
version.
b. Program a script to compare the execution time for 5 million samples.
Report the Speedup Factor (Python Time / Numba Time).
c. Why does the very first execu8on of the Numba function take slightly
longer than the second execution?

In [7]:
#Pure Python
def monte_carlo_pi_python(nsamples):
    inside = 0

    for i in range(nsamples):
        x = np.random.random()
        y = np.random.random()

        if x*x + y*y < 1:
            inside += 1

    pi = 4 * inside / nsamples
    return pi

#Numba
@jit(nopython=True)
def monte_carlo_pi_numba(nsamples):
    inside = 0

    for i in range(nsamples):
        x = np.random.random()
        y = np.random.random()

        if x*x + y*y < 1:
            inside += 1

    pi = 4 * inside / nsamples
    return pi

nsamples = 5000000

# -------- Python --------
start = time.time()
pi_py = monte_carlo_pi_python(nsamples)
python_time = time.time() - start
print("Python Time:", python_time)
print("Estimated Pi (Python):", pi_py)

# -------- Numba --------
monte_carlo_pi_numba(nsamples)

start = time.time()
pi_nb = monte_carlo_pi_numba(nsamples)
numba_time = time.time() - start
print("Numba Time:", numba_time)
print("Estimated Pi (Numba):", pi_nb)

speedup = python_time / numba_time
print("Speedup Factor:", speedup)

Python Time: 5.586140155792236
Estimated Pi (Python): 3.1402608
Numba Time: 0.05318903923034668
Estimated Pi (Numba): 3.1418088
Speedup Factor: 105.02427260624589


Because Numba uses JIT compilation. During the first execution, the function is compiled into machine code, which adds overhead. Subsequent executions reuse the compiled code and run much faster.

4. You have a 1D NumPy array represen8ng pixel intensi8es (values 0–255). You
need to increase the brightness of every pixel by 20%, but ensure no value
exceeds 255.
a. Write a func8on adjust_brightness(pixel_value) using the @vectorize
decorator.
b. Apply this func8on to an array of 10 million random integers.
c. Change the decorator to @vectorize(['int64(int64)'], target='parallel').
Measure the 8me difference when the work is automa8cally split
across your CPU cores.
d. What happens if you try to pass a list instead of a NumPy array to this
func8on?

In [8]:
from numba import vectorize
@vectorize
def adjust_brightness(pixel):
    new_val = pixel * 1.2
    if new_val > 255:
        return 255
    else:
        return int(new_val)

arr = np.random.randint(0, 256, 10000000)

start = time.time()
result = adjust_brightness(arr)
print("Vectorize Time:", time.time() - start)

@vectorize(['int64(int64)'], target='parallel')
def adjust_brightness_parallel(pixel):
    new_val = pixel * 1.2
    if new_val > 255:
        return 255
    else:
        return int(new_val)
start = time.time()
result_parallel = adjust_brightness_parallel(arr)
print("Parallel Time:", time.time() - start)

Vectorize Time: 0.13512754440307617
Parallel Time: 0.029545307159423828


Numba’s vectorize functions are designed to work efficiently with NumPy arrays. If a Python list is passed, it may either be converted internally to a NumPy array or result in slower performance because lists are not optimized for vectorized operation.

5. Write Python code to generate synthe8c training data of 100,000 samples,
10 features and binary labels {-1, +1}. Implement binary logis8c regression
using the mathema8cal formula for gradient descent:
a. Using standard NumPy (without Numba)
b. Using Numba JIT accelera8on
c. Compare correctness and performance.

In [11]:
np.random.seed(0)

n_samples = 100000
n_features = 10

X = np.random.randn(n_samples, n_features)
y = np.random.choice([-1, 1], n_samples)

# Initialize weights
w = np.zeros(n_features)
lr = 0.01
epochs = 10

#-----------Numpy------------
def logistic_numpy(X, y, w, lr, epochs):
    n = len(y)

    for e in range(epochs):
        for i in range(n):
            z = 0
            for j in range(len(w)):
                z += w[j] * X[i][j]

            # sigmoid
            pred = 1 / (1 + np.exp(-y[i]*z))

            for j in range(len(w)):
                w[j] += lr * (1 - pred) * y[i] * X[i][j]

    return w

#--------------Numba-----------------
@jit(nopython=True)
def logistic_numba(X, y, w, lr, epochs):
    n = len(y)

    for e in range(epochs):
        for i in range(n):
            z = 0
            for j in range(len(w)):
                z += w[j] * X[i][j]

            pred = 1 / (1 + np.exp(-y[i]*z))

            for j in range(len(w)):
                w[j] += lr * (1 - pred) * y[i] * X[i][j]

    return w

# -------- NumPy --------
w1 = np.zeros(n_features)
start = time.time()
w1 = logistic_numpy(X, y, w1, lr, epochs)
numpy_time = time.time() - start
print("NumPy Time:", numpy_time)

# -------- Numba --------
w2 = np.zeros(n_features)

logistic_numba(X, y, w2, lr, epochs)  # warm-up

start = time.time()
w2 = logistic_numba(X, y, w2, lr, epochs)
numba_time = time.time() - start
print("Numba Time:", numba_time)

NumPy Time: 19.207947969436646
Numba Time: 0.05045747756958008


6. Write a CUDA kernel to add two large matrices (A + B = C) of size 1024 X 1024.

In [13]:
@cuda.jit
def matrix_add(A, B, C):
    row, col = cuda.grid(2)

    if row < A.shape[0] and col < A.shape[1]:
        C[row][col] = A[row][col] + B[row][col]
n = 1024

A = np.random.random((n, n))
B = np.random.random((n, n))
C = np.zeros((n, n))

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.to_device(C)

threads_per_block = (16, 16)

blocks_per_grid_x = (n + threads_per_block[0] - 1) // threads_per_block[0]
blocks_per_grid_y = (n + threads_per_block[1] - 1) // threads_per_block[1]
blocks_per_grid = (blocks_per_grid_x, blocks_per_grid_y)
start = time.time()
matrix_add[blocks_per_grid, threads_per_block](d_A, d_B, d_C)
d_C.copy_to_host(C)
print("GPU Time:", time.time() - start)

GPU Time: 0.09398698806762695
